In [8]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic     # Importamos la función geodesic para calcular distancias entre coordenadas geográficas

# Cargamos el dataset en formato parquet
df = pd.read_parquet("../data/pois_barcelona_procesados.parquet")

# Mostramos las primeras filas del dataset para comprobar que los datos se han cargado correctamente
df.head()

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,...,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score,has_valid_source,visit_duration,tags,tags_str
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.930000,...,OSM,True,False,True,True,3.21900,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,0.503766,...,None,False,False,False,False,2.56796,False,60,"[culture, indoor]",culture|indoor
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.650000,...,OSM,True,False,True,True,3.06500,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor
3,111,Archivo Histórico de la Ciudad de Barcelona,cultural,library,41.3841,2.17567,Barcelona,archive,4.3,0.503766,...,None,False,False,False,False,2.68696,False,60,"[culture, indoor]",culture|indoor
4,112,Arxiu Diocesà de Barcelona,cultural,library,41.3837,2.17534,Barcelona,episcopal archive,4.2,0.750000,...,OSM,True,False,True,True,3.16500,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor


In [9]:
# Verificamos que estám todas las columnas necesarias para realizar el modelado
df.columns

Index(['id', 'name', 'category', 'subcategory', 'latitude', 'longitude',
       'city', 'description', 'rating', 'match_confidence', 'opening_hours',
       'opening_hours_source', 'has_opening_hours', 'is_24_7',
       'is_likely_open', 'has_match_confidence', 'score', 'has_valid_source',
       'visit_duration', 'tags', 'tags_str'],
      dtype='object')

In [10]:
# Comprobamos los tipos de las columnas para verificar que son los correctos
df.dtypes

id                         int64
name                      object
category                category
subcategory             category
latitude                 float64
longitude                float64
city                      object
description               object
rating                   float64
match_confidence         float64
opening_hours             object
opening_hours_source      object
has_opening_hours           bool
is_24_7                     bool
is_likely_open              bool
has_match_confidence        bool
score                    float64
has_valid_source            bool
visit_duration             int64
tags                      object
tags_str                  object
dtype: object

In [11]:
# Creamos un nuevo DataFrame llamado df_base seleccionando solo las columnas relevantes  para simplificar el dataset y trabajar solo con lo necesario
df_base = df[[
    'id',
    'name',
    'category',
    'subcategory',
    'latitude',
    'longitude',
    'rating',
    'score',
    'visit_duration'
]].copy()   # hacemos una copia para no modificar el dataset original

# Eliminamos filas donde falten coordenadas (latitud o longitud) o porque luego vamos a calcular distancias geográficas
# Si hay valores NaN, la función geodesic falla
df_base = df_base.dropna(subset=['latitude', 'longitude'])

# Creamos una nueva columna llamada 'score_final' la cual será la métrica que usaremos para rankear los lugares
df_base['score_final'] = (
    df_base['rating'] * 0.6 +   # damos más peso al rating (60%)
    df_base['score'] * 0.4      # combinamos con el score interno (40%)
)

In [12]:
# Comprobamos los valores de la columna de categoria para hacer pruebas posteriores
df["category"].value_counts()

category
cultural              331
monument              100
tourist_attraction     84
fountain               75
park                   61
religious              44
historic               41
administrative         35
bridge                  3
unknown                 1
Name: count, dtype: int64

In [13]:
# Esta función sirve para obtener los mejores POIs según ciertos filtros
def recomendar(df, categoria=None, min_rating=0, top_n=10):
    
    # Hacemos una copia del DataFrame para no modificar el original
    data = df.copy()
    
    # Si el usuario especifica una categoría (por ejemplo "cultural") filtramos solo los POIs de esa categoría
    if categoria:
        data = data[data['category'] == categoria]
    
    # Filtramos los POIs que tengan un rating mínimo para evitar lugares con baja calidad
    data = data[data['rating'] >= min_rating]
    
    # Ordenamos los resultados por la columna 'score_final' de mayor a menor
    # Luego nos quedamos con los primeros N resultados (top_n)
    return data.sort_values(by='score_final', ascending=False).head(top_n)

# Ejecutamos la función (PRIMERA PRUEBA):
# - Solo POIs de categoría "cultural"
# - Con rating mínimo de 4
# - Mostramos los 5 mejores
recomendar(df_base, categoria='cultural', min_rating=4, top_n=5)

,id,name,category,subcategory,latitude,longitude,rating,score,visit_duration,score_final
225,6636,Palacio de la Música Catalana,cultural,theatre,41.3876,2.17522,4.9,3.709,120,4.4236
476,335226,Centre de Documentació Begoña Raventós -Biblio...,cultural,library,41.3850,2.15221,4.9,3.544,60,4.3576
471,335216,Bolsa de Barcelona,cultural,library,41.3899,2.16718,4.8,3.594,60,4.3176
292,7595,Museo del Chocolate (Barcelona),cultural,museum,41.3871,2.18174,4.8,3.552,120,4.3008
724,336266,Cases Rocamora,cultural,museum,41.3889,2.16940,4.8,3.504,120,4.2816


In [14]:
def generar_ruta(df, start_point, n=5):
    puntos = df.copy()
    
    # eliminar coordenadas no válidas
    puntos = puntos.dropna(subset=['latitude', 'longitude'])
    
    ruta = []
    actual = start_point
    
    for _ in range(min(n, len(puntos))):
        puntos['dist'] = puntos.apply(
            lambda x: geodesic(actual, (x['latitude'], x['longitude'])).km,
            axis=1
        )
        
        siguiente = puntos.sort_values('dist').iloc[0]
        ruta.append(siguiente)
        
        actual = (siguiente['latitude'], siguiente['longitude'])
        puntos = puntos.drop(siguiente.name)
    
    return pd.DataFrame(ruta)

start = (41.3851, 2.1734)

df_cultural = df_base[
    (df_base['category'] == 'cultural') &
    (df_base['rating'] >= 4.0)
].copy()

ruta_cultural = generar_ruta(df_cultural, start, n=5)
ruta_cultural[['name', 'category', 'rating']]

,name,category,rating
536,Teatre Tirso de Molina (Barcelona),cultural,4.6
149,El mundo nace en cada beso,cultural,4.7
505,Sant Josep Oriol,cultural,4.1
22,Real Círculo Artístico de Barcelona,cultural,4.2
504,Biblioteca del Colegio de Arquitectos de Cataluña,cultural,4.8
